# ARC Prize 2026 wheelhouse

Downloads the runtime wheelhouse used by the Duck submission into
`/kaggle/working`. Run this notebook with internet enabled and a GPU image.
Downstream submissions install vLLM from this notebook output with
`--no-index --find-links`.


In [ ]:
import os, platform, subprocess, sys

print("python      :", sys.version)
print("platform    :", platform.platform())
try:
    import torch
    print("torch       :", torch.__version__)
    print("cuda avail  :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda version:", torch.version.cuda)
        print("gpu         :", torch.cuda.get_device_name(0))
except ImportError:
    print("torch       : not installed")

try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("nvidia-smi  :", r.stdout.strip())
except FileNotFoundError:
    pass
print("kaggle env  :", {k: v for k, v in os.environ.items() if k.startswith("KAGGLE")})

In [ ]:
import subprocess, sys
from pathlib import Path

dest = Path("/kaggle/working")

# These pins mirror the setup stamp in Tufa's public share bundle. If Kaggle's
# image changes underneath us, edit this list and re-run the wheels notebook.
requirements = dest / "requirements.lock"
requirements.write_text(
    "\n".join([
        "vllm==0.19.1",
        "torch==2.10.0",
        "flashinfer-python==0.6.6",
        "transformers",
        "accelerate",
        "peft",
        "pillow",
        "numpy",
        "scipy",
        "matplotlib",
        "requests",
        "python-dotenv",
        "imageio",
        "imageio-ffmpeg",
    ]) + "\n",
    encoding="utf-8",
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "download",
        "--requirement", str(requirements),
        "--extra-index-url", "https://download.pytorch.org/whl/cu128",
        "-d", str(dest),
    ],
    check=True,
)

wheels = sorted(dest.glob("*.whl"))
print(f"{len(wheels)} wheels downloaded")
assert any(w.name.startswith("vllm-") for w in wheels), "no vLLM wheel found"
subprocess.run(["du", "-sh", str(dest)], check=True)